# Day 083 — Exercise 5: PlannerAgent

**What you'll build:** `PlannerAgent` — an agent that encapsulates planning, sorting, and execution, with separate `plan()`, `execute()`, and `run()` methods so you can inspect the plan before committing to it.

**Why it matters:** `plan()` lets you review and approve the decomposition before anything executes — an early form of human-in-the-loop that Day 87 will formalise as a proper guardrail. And separating plan from execute means you can inject a manually-crafted task list or revise the plan before running.

In [ ]:
import json

_PLAN_JSON = json.dumps([
    {'id': 't1', 'title': 'Gather facts',
     'description': 'Collect the relevant information.', 'depends_on': []},
    {'id': 't2', 'title': 'Draft outline',
     'description': 'Organize the facts into an outline.', 'depends_on': ['t1']},
    {'id': 't3', 'title': 'Write summary',
     'description': 'Write the final summary.', 'depends_on': ['t2']},
])

def _mock_planner(plan_json=None, task_result='Task done.'):
    """Return an llm_fn: the plan JSON on planning calls, task_result on execution calls."""
    plan = plan_json if plan_json is not None else _PLAN_JSON
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'json array' in system.lower() or 'planning' in system.lower():
            return plan
        return task_result
    return _fn

def _mock_executor(task):
    return 'Result: ' + task.title
import json
from dataclasses import dataclass, field

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, list) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the Task dataclass ────────────────────────────────────────────────────────
@dataclass
class Task:
    """One step in a plan.

    Attributes:
        id:          short snake_case identifier (e.g. 't1', 'write_outline').
        title:       brief human-readable label (5 words max).
        description: one sentence describing what to do.
        depends_on:  ids of tasks that must complete before this one.
        status:      'pending' | 'done' | 'failed'.
        result:      the output of executing this task.
    """
    id: str
    title: str
    description: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    result: str = ""

# ── planning: ask the LLM to decompose a goal ─────────────────────────────────
def build_plan_prompt(goal, context=None):
    """Build a prompt that asks the LLM to break a goal into a JSON task list."""
    system = "\n".join([
        "You are a planning assistant. Break the goal into an ordered list of tasks.",
        "",
        "Return ONLY a JSON array. Each item must have these exact keys:",
        '  "id": short snake_case identifier (t1, t2, ...)',
        '  "title": brief label (5 words max)',
        '  "description": one sentence - what to do',
        '  "depends_on": list of task ids that must finish before this one ([] if none)',
        "",
        "Return ONLY the JSON array. No prose, no markdown fences.",
    ])
    user_parts = ["Goal: " + str(goal)]
    if context:
        user_parts.append("Context: " + str(context))
    return [{"role": "system", "content": system},
            {"role": "user", "content": "\n".join(user_parts)}]


def parse_plan(text):
    """Extract a task list from LLM output. Returns list[Task]; never raises.

    Tolerates markdown fences, prose before/after, missing fields, and invalid
    JSON. Invalid or missing fields are filled with safe defaults so any
    parseable item becomes a valid Task.
    """
    items = safe_parse_list(text) or []
    tasks = []
    for i, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        tasks.append(Task(
            id=str(item.get("id", "t" + str(i + 1))),
            title=str(item.get("title", "Task " + str(i + 1))),
            description=str(item.get("description", "")),
            depends_on=[str(d) for d in item.get("depends_on", [])
                        if isinstance(d, str)],
        ))
    return tasks

# ── dependency ordering: Kahn's topological sort ──────────────────────────────
def topo_sort(tasks):
    """Sort tasks so every dependency comes before the task that needs it.

    Uses Kahn's algorithm (BFS on a DAG). If a cycle exists the cyclic tasks
    are appended at the end in their original order rather than raising, so
    execution can still proceed on the non-cyclic portion.
    """
    by_id = {t.id: t for t in tasks}
    # count incoming edges (how many unresolved deps each task has)
    in_deg = {t.id: 0 for t in tasks}
    for t in tasks:
        for dep in t.depends_on:
            if dep in in_deg:
                in_deg[t.id] += 1
    # start with tasks that have no deps
    queue = [t.id for t in tasks if in_deg[t.id] == 0]
    order = []
    while queue:
        tid = queue.pop(0)
        order.append(by_id[tid])
        # for every task that listed tid as a dep, reduce its in-degree
        for t in tasks:
            if tid in t.depends_on:
                in_deg[t.id] -= 1
                if in_deg[t.id] == 0:
                    queue.append(t.id)
    # cycle guard: any task not yet emitted has a circular dependency
    done_ids = {t.id for t in order}
    for t in tasks:
        if t.id not in done_ids:
            order.append(t)
    return order

# ── executing tasks ───────────────────────────────────────────────────────────
def build_execution_context(task, results):
    """Render prior results that this task depends on, for injection into the prompt."""
    lines = ["You are executing one step of a multi-task plan."]
    prior = [(dep, results[dep]) for dep in task.depends_on if dep in results]
    if prior:
        lines.append("Results from earlier steps:")
        for dep_id, res in prior:
            lines.append("  " + dep_id + ": " + str(res))
    lines.append("Task: " + task.title)
    lines.append("Description: " + task.description)
    lines.append("Complete this task concisely.")
    return "\n".join(lines)


def execute_task(task, results, executor_fn=None, llm_fn=None):
    """Run one task. Returns the result string; never raises.

    If executor_fn is provided, call executor_fn(task) -> str.
    Otherwise use the LLM, passing prior results as context.
    Exceptions are caught and returned as 'Error: ...' strings.
    """
    try:
        if executor_fn is not None:
            return str(executor_fn(task))
        context = build_execution_context(task, results)
        messages = [{"role": "system", "content": context},
                    {"role": "user", "content": "Execute this task now."}]
        return call_llm(messages, llm_fn=llm_fn)
    except Exception as exc:
        return "Error: " + str(exc)

# ── end-to-end plan runner ────────────────────────────────────────────────────
def run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20):
    """Plan a goal, sort by dependencies, and execute step by step.

    Returns {"tasks": list[Task], "results": {id: result}, "answer": str}.
    The answer is the result of the last task in execution order.
    max_tasks caps the plan so a model that returns 1000 tasks cannot hang the gate.
    """
    plan_text = call_llm(build_plan_prompt(goal), llm_fn=llm_fn)
    tasks = parse_plan(plan_text)
    if not tasks:
        return {"tasks": [], "results": {}, "answer": "No plan generated."}
    ordered = topo_sort(tasks[:max_tasks])
    results = {}
    for task in ordered:
        result = execute_task(task, results,
                              executor_fn=executor_fn, llm_fn=llm_fn)
        task.result = result
        task.status = "done"
        results[task.id] = result
    answer = ordered[-1].result if ordered else "No tasks executed."
    return {"tasks": ordered, "results": results, "answer": answer}


## Task

`PlannerAgent(executor_fn=None, llm_fn=None, max_tasks=20)`

1. `plan(goal)` — `call_llm(build_plan_prompt(goal))` → `parse_plan` → `topo_sort`; cap at `max_tasks`; return the sorted list **without** executing.
2. `execute(tasks)` — `topo_sort(tasks[:max_tasks])`; loop calling `execute_task`; update `task.result`/`task.status`; return `{id: result}`.
3. `run(goal)` — `plan` + `execute`; build the result dict; append to `_history`; return it.
4. `history()` — copy; `clear_history()` — in-place.

## Your Implementation

In [ ]:
class PlannerAgent:
    """An agent that breaks a goal into subtasks and executes them in order."""

    def __init__(self, executor_fn=None, llm_fn=None, max_tasks=20):
        raise NotImplementedError

    def plan(self, goal):
        """Decompose goal into a sorted task list; do not execute."""
        raise NotImplementedError

    def execute(self, tasks):
        """Execute a task list in dependency order. Returns {id: result}."""
        raise NotImplementedError

    def run(self, goal):
        """Plan + execute; records in history. Returns {goal,tasks,results,answer}."""
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:

# ── the planning assistant ────────────────────────────────────────────────────
class PlannerAgent:
    """An agent that breaks a goal into subtasks and executes them in order.

    plan() decomposes a goal without executing - useful for reviewing the plan.
    execute() runs a task list produced by plan() (or built manually).
    run() does both in one call and records the outcome in history.

    Example::

        agent = PlannerAgent(executor_fn=my_executor, llm_fn=my_llm_fn)
        result = agent.run("Write a short report on prompt engineering")
        print(result["answer"])
    """

    def __init__(self, executor_fn=None, llm_fn=None, max_tasks=20):
        self._executor_fn = executor_fn
        self._llm_fn = llm_fn
        self.max_tasks = max_tasks
        self._history = []

    def plan(self, goal):
        """Decompose goal into a topologically sorted task list; do not execute."""
        plan_text = call_llm(build_plan_prompt(goal), llm_fn=self._llm_fn)
        return topo_sort(parse_plan(plan_text)[:self.max_tasks])

    def execute(self, tasks):
        """Execute a task list in dependency order. Returns {id: result} dict."""
        ordered = topo_sort(tasks[:self.max_tasks])
        results = {}
        for task in ordered:
            result = execute_task(task, results,
                                  executor_fn=self._executor_fn,
                                  llm_fn=self._llm_fn)
            task.result = result
            task.status = "done"
            results[task.id] = result
        return results

    def run(self, goal):
        """Plan + execute. Records the run in history and returns the result dict."""
        tasks = self.plan(goal)
        results = self.execute(tasks)
        answer = tasks[-1].result if tasks else "No tasks."
        record = {"goal": goal, "tasks": tasks, "results": results, "answer": answer}
        self._history.append(record)
        return record

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear run history in place."""
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 6
try:
    agent = PlannerAgent(executor_fn=_mock_executor,
                         llm_fn=_mock_planner())

    tasks = agent.plan('test goal')
    assert len(tasks) == 3 and tasks[0].id == 't1'
    score += 1; print("✅ plan() returns sorted tasks without executing them")

    assert all(t.status == 'pending' for t in tasks)
    score += 1; print("✅ plan() leaves tasks pending (no execution)")

    results = agent.execute(tasks)
    assert results['t1'] == 'Result: Gather facts'
    assert tasks[0].status == 'done'
    score += 1; print("✅ execute() runs tasks in order and marks them done")

    record = agent.run('another goal')
    assert 'goal' in record and 'tasks' in record and 'answer' in record
    score += 1; print("✅ run() returns the full result dict")

    assert len(agent.history()) == 1
    agent.history().clear()
    assert len(agent.history()) == 1
    score += 1; print("✅ history() returns a copy, not the live list")

    agent.clear_history()
    assert len(agent.history()) == 0
    score += 1; print("✅ clear_history empties the log")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the planning assistant ────────────────────────────────────────────────────
class PlannerAgent:
    """An agent that breaks a goal into subtasks and executes them in order.

    plan() decomposes a goal without executing - useful for reviewing the plan.
    execute() runs a task list produced by plan() (or built manually).
    run() does both in one call and records the outcome in history.

    Example::

        agent = PlannerAgent(executor_fn=my_executor, llm_fn=my_llm_fn)
        result = agent.run("Write a short report on prompt engineering")
        print(result["answer"])
    """

    def __init__(self, executor_fn=None, llm_fn=None, max_tasks=20):
        self._executor_fn = executor_fn
        self._llm_fn = llm_fn
        self.max_tasks = max_tasks
        self._history = []

    def plan(self, goal):
        """Decompose goal into a topologically sorted task list; do not execute."""
        plan_text = call_llm(build_plan_prompt(goal), llm_fn=self._llm_fn)
        return topo_sort(parse_plan(plan_text)[:self.max_tasks])

    def execute(self, tasks):
        """Execute a task list in dependency order. Returns {id: result} dict."""
        ordered = topo_sort(tasks[:self.max_tasks])
        results = {}
        for task in ordered:
            result = execute_task(task, results,
                                  executor_fn=self._executor_fn,
                                  llm_fn=self._llm_fn)
            task.result = result
            task.status = "done"
            results[task.id] = result
        return results

    def run(self, goal):
        """Plan + execute. Records the run in history and returns the result dict."""
        tasks = self.plan(goal)
        results = self.execute(tasks)
        answer = tasks[-1].result if tasks else "No tasks."
        record = {"goal": goal, "tasks": tasks, "results": results, "answer": answer}
        self._history.append(record)
        return record

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear run history in place."""
        self._history.clear()
```

**Why separate `plan` and `execute`?** You might want to inspect — or even edit — the task list before running it. Keeping them separate costs nothing and gives you the hook you need for human review. Day 87 will put an approval gate right at this seam.

</details>